# Notebook 5: Meridian Post-Modeling — Diagnostics & Budget Optimization

This notebook picks up **after** fitting a Meridian model (Session 5) and walks
through the full post-modeling workflow:

1. **Model Health** — automated diagnostics and health score
2. **Model Fit Assessment** — predictive accuracy, baseline probability
3. **Built-in Visualizations** — Meridian's native Altair plots
4. **Budget Optimization** — fixed and flexible budget scenarios
5. **Scenario Planning** — what-if analysis with modified spend
6. **HTML Reports** — one-page summaries for model results and optimization

**Prerequisites:** A fitted Meridian model. We re-fit the model from Session 5
in the setup cells below so this notebook is self-contained.

**Reference:** [Meridian Post-Modeling Docs](https://developers.google.com/meridian/docs/post-modeling/intro)

In [ ]:
# --- Setup & Model Fitting (recap from Session 5) ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Meridian imports
from meridian.data.input_data import InputData
from meridian.model import model as meridian_model
from meridian.model import spec as meridian_spec
from meridian.analysis.analyzer import Analyzer, DataTensors
from meridian.analysis.optimizer import BudgetOptimizer
from meridian.analysis import visualizer
from meridian.analysis.review import reviewer
from meridian.analysis.summarizer import Summarizer
import meridian
import xarray as xr

print(f"Meridian version: {meridian.__version__}")

In [ ]:
# --- Load and prepare data (same as Session 5) ---
df = pd.read_excel('../../data/MMM_Workshop_Data.xlsx', sheet_name='Data')

date_col = 'Month'
kpi_col = 'Sales_Revenue_Total'

media_cols = [col for col in df.columns if 'Spend' in col or 'spend' in col]
drop_cols = ['Meta_Spends_Agg']  # collinear with Meta1 + Meta2
media_cols = [c for c in media_cols if c not in drop_cols]

n_times = len(df)
n_geos = 1
n_media = len(media_cols)

media_spend_array = df[media_cols].values.reshape(n_geos, n_times, n_media)
media_volume_array = media_spend_array.copy()
kpi_array = df[kpi_col].values.reshape(n_geos, n_times)

time_coords = [d.strftime('%Y-%m-%d') for d in pd.to_datetime(df[date_col])]
geo_coords = ['national']

input_data = InputData(
    kpi=xr.DataArray(
        kpi_array, name='kpi',
        dims=['geo', 'time'],
        coords={'time': time_coords, 'geo': geo_coords}
    ),
    kpi_type='revenue',
    population=xr.DataArray(
        [1.0], name='population',
        dims=['geo'],
        coords={'geo': geo_coords}
    ),
    media_spend=xr.DataArray(
        media_spend_array, name='media_spend',
        dims=['geo', 'time', 'media_channel'],
        coords={'time': time_coords, 'geo': geo_coords, 'media_channel': media_cols}
    ),
    media=xr.DataArray(
        media_volume_array, name='media',
        dims=['geo', 'media_time', 'media_channel'],
        coords={'media_time': time_coords, 'geo': geo_coords, 'media_channel': media_cols}
    ),
)

print(f"Dataset: {n_times} periods, {n_media} channels")
print(f"Channels: {media_cols}")

In [ ]:
# --- Fit the model ---
model_spec = meridian_spec.ModelSpec(max_lag=8)

mmm = meridian_model.Meridian(
    input_data=input_data,
    model_spec=model_spec,
)

print("Fitting model (5-15 min on CPU)...")
mmm.sample_posterior(
    n_chains=2, n_adapt=500, n_burnin=500, n_keep=500, seed=0
)
mmm.sample_prior(n_draws=500, seed=0)
print("Done!")

---
## 1. Model Health Checks

Meridian provides an automated **ModelReviewer** that runs six diagnostic checks
and returns a PASS / REVIEW / FAIL status for each:

| Check | What it tests | Pass threshold |
|-------|---------------|----------------|
| Convergence | Max R-hat across parameters | < 1.2 |
| Negative Baseline | Probability baseline stays non-negative | > 0.8 |
| Bayesian PPP | Posterior predictive p-value | ≥ 0.05 |
| Goodness-of-Fit | R², MAPE, wMAPE | R² mapping via sigmoid |
| Prior-Posterior Shift | Whether data updated ROI estimates | Low failure rate |
| ROI Consistency | Posterior means in reasonable prior range | Low failure rate |

The checks are combined into a **health score** (0-100) using weighted averaging:
- Negative Baseline: 30%, Bayesian PPP: 30%, GoF: 10%, Prior-Posterior Shift: 15%, ROI Consistency: 15%
- Convergence is a gate: if it fails, the score is 0.

Scores ≥ 90 suggest a robust model. Scores ≤ 70 warrant closer examination.

In [ ]:
# --- Run Model Health Checks ---
model_reviewer = reviewer.ModelReviewer(mmm)
health_summary = model_reviewer.run()

# Print the summary (shows each check's status)
print(health_summary)

# Access individual check statuses programmatically
print("\n=== Check Statuses ===")
for check_name, status in health_summary.checks_status.items():
    print(f"  {check_name}: {status}")

---
## 2. Model Fit Assessment

Beyond health checks, we can assess model quality through:

- **Predictive accuracy** — R², MAPE, and wMAPE comparing expected vs actual KPI
- **Negative baseline probability** — how likely is the model to predict negative
  outcomes when all media is turned off? (Should be low.)
- **Expected vs actual** — visual comparison of model predictions to reality

> **Important:** The goal in MMM is *causal inference*, not minimizing prediction
> error. A model with slightly lower R² but better-calibrated priors may give
> more reliable ROI estimates than an overfit model.

In [ ]:
# --- Predictive Accuracy ---
analyzer = Analyzer(mmm)

accuracy = analyzer.predictive_accuracy()
print("=== Predictive Accuracy ===")
print(accuracy)
print()

# Formatted table via the visualizer
diag = visualizer.ModelDiagnostics(mmm)
accuracy_table = diag.predictive_accuracy_table()
print("=== Formatted Accuracy Table ===")
print(accuracy_table)

In [ ]:
# --- Negative Baseline Probability ---
neg_baseline_prob = analyzer.negative_baseline_probability()
print(f"Probability of negative baseline: {neg_baseline_prob:.4f}")
print()

if neg_baseline_prob > 0.2:
    print("WARNING: High negative baseline probability.")
    print("This suggests the model may over-attribute revenue to media,")
    print("leaving an unrealistically low (or negative) baseline.")
    print("Consider adding control variables or revising priors.")
else:
    print("Baseline looks reasonable — low probability of going negative.")

In [ ]:
# --- Expected vs Actual: Model Fit Plot ---
# Meridian's built-in Altair chart
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit(include_baseline=True, include_ci=True)

### Interpreting the Model Fit Plot

- **Blue line**: Model's expected outcome (posterior mean)
- **Shaded band**: Credible interval
- **Black dots/line**: Actual observed KPI
- **Gray area**: Estimated baseline (outcome without any media)

A good fit means the blue line tracks the actual data closely, and the actuals
fall within the credible interval most of the time. The baseline should be
positive and represent a plausible "no marketing" scenario.

---
## 3. Built-in Visualizations

Meridian ships with rich Altair-based visualizations via the `visualizer` module.
These are interactive — hover over data points for details.

### Visualizer Classes

| Class | Key Plots |
|-------|----------|
| `ModelDiagnostics` | R-hat boxplot, prior/posterior distributions |
| `ModelFit` | Expected vs actual outcome |
| `MediaEffects` | Response curves, adstock decay, Hill curves |
| `MediaSummary` | ROI bars, contribution waterfall/pie, spend vs contribution |

In [ ]:
# --- R-hat Convergence Diagnostics ---
diag = visualizer.ModelDiagnostics(mmm)
diag.plot_rhat_boxplot()

In [ ]:
# --- Prior vs Posterior Distributions (ROI) ---
# Shows whether the data updated the model's beliefs about ROI
diag.plot_prior_and_posterior_distribution(parameter='roi_m')

In [ ]:
# --- Response Curves (Altair) ---
media_effects = visualizer.MediaEffects(mmm)
media_effects.plot_response_curves(include_ci=True)

In [ ]:
# --- Adstock Decay Curves ---
# Shows how quickly media effects decay over time for each channel
media_effects.plot_adstock_decay(include_ci=True)

In [ ]:
# --- Hill Saturation Curves ---
# Shows diminishing returns as media units increase
# Returns a dict keyed by channel type
hill_charts = media_effects.plot_hill_curves(include_prior=True, include_ci=True)
for channel_type, chart in hill_charts.items():
    print(f"\nChannel type: {channel_type}")
    display(chart)

In [ ]:
# --- Media Summary Visualizations ---
media_summary = visualizer.MediaSummary(mmm)

# ROI by channel with credible intervals
media_summary.plot_roi_bar_chart(include_ci=True)

In [ ]:
# Contribution waterfall chart
media_summary.plot_contribution_waterfall_chart()

In [ ]:
# Spend share vs contribution share
media_summary.plot_spend_vs_contribution()

In [ ]:
# ROI vs Effectiveness (bubble chart — size = spend)
media_summary.plot_roi_vs_effectiveness()

In [ ]:
# ROI vs Marginal ROI — identifies channels with headroom
# Channels above the diagonal have mROI > ROI → still growing efficiently
# Channels below the diagonal are saturated → mROI < ROI
media_summary.plot_roi_vs_mroi()

In [ ]:
# Summary metrics table (formatted with credible intervals)
summary_table = media_summary.summary_table(
    include_prior=True, include_posterior=True
)
print("=== Summary Metrics Table ===")
summary_table

---
## 4. Budget Optimization

Meridian's `BudgetOptimizer` uses the fitted response curves to find the
allocation that **maximizes total outcome** given budget constraints.

### Two Optimization Modes

| Mode | What you fix | What the optimizer finds |
|------|-------------|------------------------|
| **Fixed budget** | Total budget | Optimal allocation across channels |
| **Flexible budget** | Target ROI or mROI | Optimal total budget + allocation |

### Spend Constraints

You can set per-channel bounds to keep the optimizer realistic:
- `spend_constraint_lower=0.3` → each channel gets at least 70% of its current spend
- `spend_constraint_upper=0.3` → each channel gets at most 130% of its current spend

Wider constraints give the optimizer more freedom but may produce impractical
recommendations (e.g., "put everything in one channel").

In [ ]:
# --- Fixed Budget Optimization ---
budget_optimizer = BudgetOptimizer(mmm)

# Use historical total spend as the budget
historical_spend = float(media_spend_array.sum())
print(f"Historical total spend: ${historical_spend:,.0f}")

# Optimize: same total budget, find best allocation
opt_results = budget_optimizer.optimize(
    fixed_budget=True,
    budget=historical_spend,
    spend_constraint_lower=0.3,  # min 70% of current per channel
    spend_constraint_upper=0.3,  # max 130% of current per channel
)

print("\nOptimization complete!")

In [ ]:
# --- Compare Non-Optimized vs Optimized ---
print("=== Non-Optimized (Historical) ===")
print(opt_results.nonoptimized_data)

print("\n=== Optimized ===")
print(opt_results.optimized_data)

In [ ]:
# --- Optimization Visualizations ---
# Spend reallocation: how much shifts to/from each channel
opt_results.plot_spend_delta()

In [ ]:
# Incremental outcome change from reallocation
opt_results.plot_incremental_outcome_delta()

In [ ]:
# Optimized budget allocation (pie chart)
opt_results.plot_budget_allocation(optimized=True)

In [ ]:
# Response curves with spend constraint bounds highlighted
opt_results.plot_response_curves()

---
### Flexible Budget Optimization

Instead of fixing the budget, you can set a **target ROI** or **target mROI**
and let the optimizer find the right total budget level.

- `target_roi`: "I want at least X return per dollar across all channels"
- `target_mroi`: "I want the marginal return of the last dollar to be at least X"

In [ ]:
# --- Flexible Budget: Target ROI ---
opt_flex = budget_optimizer.optimize(
    fixed_budget=False,
    target_roi=1.5,  # want at least $1.50 back per $1 spent
    spend_constraint_lower=0.3,
    spend_constraint_upper=0.3,
)

print("=== Flexible Budget Optimization (target ROI = 1.5) ===")
print(f"Optimized total budget: ${float(opt_flex.optimized_data.attrs['budget']):,.0f}")
print(f"Historical budget:      ${historical_spend:,.0f}")
print(f"\nOptimized total ROI:    {float(opt_flex.optimized_data.attrs['total_roi']):.2f}")
print()
print(opt_flex.optimized_data)

---
## 5. Scenario Planning

Beyond simple optimization, you can run **what-if scenarios** by passing
modified data via `DataTensors`. This lets you answer questions like:

- What if we **double TV spend** but keep everything else the same?
- What if **cost per impression** changes for digital channels?
- What if we shift spend from one **time period** to another?

The optimizer re-runs with your hypothetical data, giving you a counterfactual
comparison.

In [ ]:
# --- Scenario: What if we increase total budget by 20%? ---
increased_budget = historical_spend * 1.2

opt_increased = budget_optimizer.optimize(
    fixed_budget=True,
    budget=increased_budget,
    spend_constraint_lower=0.3,
    spend_constraint_upper=0.3,
)

print("=== Scenario: +20% Total Budget ===")
print(f"Budget: ${increased_budget:,.0f} (was ${historical_spend:,.0f})")
print(f"\nOptimized total outcome: "
      f"${float(opt_increased.optimized_data.attrs['total_incremental_outcome']):,.0f}")
print(f"Baseline total outcome:  "
      f"${float(opt_results.optimized_data.attrs['total_incremental_outcome']):,.0f}")

opt_increased.plot_spend_delta()

In [ ]:
# --- Scenario: What if we cut budget by 20%? ---
decreased_budget = historical_spend * 0.8

opt_decreased = budget_optimizer.optimize(
    fixed_budget=True,
    budget=decreased_budget,
    spend_constraint_lower=0.5,  # tighter constraints — can't cut more than 50%
    spend_constraint_upper=0.5,
)

print("=== Scenario: -20% Total Budget ===")
print(f"Budget: ${decreased_budget:,.0f} (was ${historical_spend:,.0f})")
print(f"\nOptimized total outcome: "
      f"${float(opt_decreased.optimized_data.attrs['total_incremental_outcome']):,.0f}")

opt_decreased.plot_spend_delta()

In [ ]:
# --- Compare Scenarios Side by Side ---
scenarios = {
    '-20% Budget': opt_decreased,
    'Current (Optimized)': opt_results,
    '+20% Budget': opt_increased,
}

comparison = []
for name, result in scenarios.items():
    data = result.optimized_data
    comparison.append({
        'Scenario': name,
        'Total Budget': float(data.attrs['budget']),
        'Total Incremental Outcome': float(data.attrs['total_incremental_outcome']),
        'Total ROI': float(data.attrs['total_roi']),
    })

comparison_df = pd.DataFrame(comparison)
print("=== Scenario Comparison ===")
print(comparison_df.to_string(index=False))

---
## 6. HTML Reports

Meridian generates self-contained HTML reports that you can share with
stakeholders. Two types are available:

1. **Model Results Summary** — model fit, contributions, ROI, response curves
2. **Optimization Summary** — budget reallocation recommendations with charts

In [ ]:
# --- Model Results HTML Report ---
summarizer_instance = Summarizer(mmm)

start_date = time_coords[0]
end_date = time_coords[-1]

summarizer_instance.output_model_results_summary(
    filename='model_results_summary.html',
    filepath='.',
    start_date=start_date,
    end_date=end_date,
)
print(f"Model results report saved to: ./model_results_summary.html")
print(f"Date range: {start_date} to {end_date}")

In [ ]:
# --- Optimization HTML Report ---
opt_results.output_optimization_summary(
    filename='optimization_summary.html',
    filepath='.',
)
print("Optimization report saved to: ./optimization_summary.html")
print("Open in a browser to see budget reallocation recommendations.")

---
## Exercises

1. **Tighten constraints**: Re-run the fixed budget optimization with
   `spend_constraint_lower=0.1` and `spend_constraint_upper=0.1`. How does
   the optimal allocation change compared to the wider 0.3 constraints?

2. **Find the break-even budget**: Use flexible budget with `target_roi=1.0`.
   What total budget gives exactly $1 return per $1 spent?

3. **Channel deep-dive**: Pick the channel with the highest mROI and the one
   with the lowest. What do their response curves look like? Does the
   adstock decay pattern explain the difference?

4. **Scenario planning**: Create a scenario where you cut the bottom 3 channels
   by 50% and redistribute that budget to the top 3 channels. Does total
   outcome increase?

In [ ]:
# TODO: Exercise 1 — Tighten constraints
# opt_tight = budget_optimizer.optimize(
#     fixed_budget=True,
#     budget=historical_spend,
#     spend_constraint_lower=0.1,
#     spend_constraint_upper=0.1,
# )
# opt_tight.plot_spend_delta()

In [ ]:
# TODO: Exercise 2 — Break-even budget
# opt_breakeven = budget_optimizer.optimize(
#     fixed_budget=False,
#     target_roi=1.0,
# )
# print(f"Break-even budget: ${float(opt_breakeven.optimized_data.attrs['budget']):,.0f}")

In [ ]:
# TODO: Exercise 3 — Channel deep-dive
# Use media_effects.plot_response_curves() and media_effects.plot_adstock_decay()
# Compare the high-mROI and low-mROI channels

In [ ]:
# TODO: Exercise 4 — Custom reallocation scenario
# Identify top 3 and bottom 3 by ROI, then create a custom pct_of_spend

---
## Key Takeaways

1. **Always run health checks** before trusting model outputs. A low health
   score signals convergence, fit, or prior specification issues.

2. **Negative baseline probability** is a critical sanity check — if the model
   says revenue would be negative without marketing, the media effects are
   over-attributed.

3. **ROI vs mROI** tells different stories:
   - ROI = how efficient was past spend (backward-looking)
   - mROI = how efficient is the next dollar (forward-looking)
   - Optimize future budgets using mROI, not historical ROI.

4. **Spend constraints matter** — unconstrained optimization often produces
   unrealistic recommendations. Use domain knowledge to set bounds.

5. **Scenario planning** bridges the model and business decisions. Run multiple
   what-if scenarios to understand the sensitivity of outcomes to budget changes.